# 03 - LLM-as-annotator: turning unlabeled reviews into a dataset

`hotel_reviews.csv` has **no labels**. This notebook shows the modern bootstrap
workflow: prompt a base (non-fine-tuned) LLM to label reviews, keep only
high-confidence predictions, and inspect agreement - the cheap first 90% of a
labeling pipeline, before humans verify a sample.

Works on CPU with the 0.5B model (slow but fine for 200 rows); T4 recommended.

## Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes
import pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

own = pd.read_csv("data/raw/hotel_reviews.csv").sample(200, random_state=42)
own = own[own["Review"].str.len() > 80].reset_index(drop=True)
print(len(own), "reviews to annotate")

## Load Qwen2.5-0.5B-Instruct (4-bit)

In [ ]:
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto").eval()

## Prompt for labels *and* confidence in one shot

In [ ]:
SYSTEM = ("You classify hotel review sentiment. Reply with JSON: "
          '{\"label\": \"positive\" | \"negative\", \"confidence\": 0.0-1.0}')

In [ ]:
import json, re

def annotate(text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Review: {text[:1500]}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=24, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    reply = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", reply, re.S)
    try:
        parsed = json.loads(m.group())
    except Exception:
        return {"label": None, "confidence": 0.0}
    label = parsed.get("label")
    return {"label": label if label in ("positive", "negative") else None,
            "confidence": float(parsed.get("confidence", 0.0))}

results = [annotate(t) for t in own["Review"]]
own["ann_label"] = [r["label"] for r in results]
own["ann_conf"] = [r["confidence"] for r in results]
own[["Review", "ann_label", "ann_conf"]].head(8)

## Confidence filtering - the key production trick

In [ ]:
# Keep only confident annotations; the rest goes to the human review queue
conf = own[own["ann_conf"] >= 0.8]
print(f"kept {len(conf)}/{len(own)} annotations above 0.8 confidence")
conf["ann_label"].value_counts().plot(kind="barh", figsize=(4, 2),
                                      title="auto-labels (confidence >= 0.8)");

In [ ]:
# Spot-check the noisiest predictions first - exactly what a human reviewer gets
own.sort_values("ann_conf").head(5)[["Review", "ann_label", "ann_conf"]]

## Where this goes next
1. Human-verify a 10% sample -> estimate true annotation accuracy (measure, don't assume).
2. Merge verified rows into `data/processed/train.parquet` and retrain any model in the benchmark.
3. Same pattern powers the "weak supervision -> distillation" upgrade path (see DESIGN.md roadmap).